# Video Generation Architectures

**Module:** 18 — Video Generation

Architecture families for video: 3D U-Nets, factorized attention, DiT-over-time, autoregressive tokens, cascaded stacks — and scaling challenges.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compare major architecture families for video generation
- Explain training signals (text–video, image–video, joint audio)
- Identify scaling challenges: memory, datasets, evaluation
- Map architecture choice to latency and controllability needs


## Architecture families

```mermaid
flowchart TB
  subgraph fam [Families]
    U[3D / inflated U-Net diffusion]
    F[Factorized space-time attention]
    D[DiT / transformer latent video]
    A[Autoregressive frame/token models]
    C[Cascaded / multi-stage]
  end
  U --> OUT[Pixels / latents]
  F --> OUT
  D --> OUT
  A --> OUT
  C --> OUT
```

| Family | Idea | Trade-off |
|--------|------|-----------|
| Inflated 3D U-Net | Extend image U-Net with temporal conv/attn | Good baseline; heavy VRAM |
| Factorized attention | Space then time (or vice versa) | Cheaper than full 3D attn |
| DiT-style | Transformer backbone on spatiotemporal tokens | Scales well; costly |
| Autoregressive | Next-frame / next-token prediction | Streaming potential; error accumulation |
| Cascaded | Base low-res → temporal/spatial super-res | Complex stacks; strong quality |


## Spatiotemporal Diffusion

### Definition
Video diffusion denoises a volume (or latent volume) across space and time, often with text conditioning via cross-attention.

### Why it matters
Dominant paradigm for many frontier and open video systems.

### How it works
Encode video to latents → add noise → predict noise with spatiotemporal net → decode frames (sometimes with temporal decoder tricks).

### Intuition
Cleaning a flipbook that was dropped in dust — page by page, but aware of neighbors.

### Pitfalls
- Naïve per-frame diffusion → flicker
- Ignoring latent temporal stride mismatches

### When to use
General txt2video/img2video systems.


In [ ]:
# Demo 1: token/memory scaling sketch
def tokens(frames, h, w, patch=16, time_patch=1):
    return (frames // time_patch) * (h // patch) * (w // patch)

def attn_cost(n_tokens):
    return n_tokens ** 2

for frames in [1, 8, 24]:
    n = tokens(frames, 512, 512)
    print(frames, "tokens", n, "attn_units", attn_cost(n))


In [ ]:
# Demo 2: factorized vs full attention cost
def full_cost(f, s):
    return (f * s) ** 2

def factorized_cost(f, s):
    # space attn per frame + time attn per spatial site (simplified)
    return f * (s ** 2) + s * (f ** 2)

s = (512//16) * (512//16)
for f in [8, 16, 32]:
    print(f, "full", full_cost(f, s), "fact", factorized_cost(f, s), "ratio", round(full_cost(f,s)/factorized_cost(f,s), 1))


## Autoregressive and Cascaded Stacks

### Definition
AR models predict the next frame or discrete token; cascaded systems generate coarse video then upsample spatially/temporally.

### Why it matters
AR can stream; cascades reclaim quality like image Imagen-style stacks.

### How it works
Train stage-1 for layout/motion; stage-2 for detail; optional audio stage. For AR, use teacher forcing + scheduled sampling carefully.

### Intuition
Rough animatic → clean-up animation → ink/paint.

### Pitfalls
- AR drift over long rollouts
- Cascade misalignment between stages

### When to use
Streaming previews, long-form research, high-res upgraders.


### Training signals

| Signal | Role |
|--------|------|
| Text–video pairs | Semantic control |
| Image–video | Img2video / keyframe fidelity |
| Video–video | Compression, restyle, prediction |
| Joint audio–video | Lip sync / soundtrack grounding |
| Synthetic renderings | Exact camera/pose supervision |

### Scaling challenges
- **Memory:** activations scale with frames × resolution
- **Data:** high-quality captioned video is expensive/noisy
- **Eval:** FVD/CLIPSIM imperfect; humans still gold
- **Safety:** temporal policy labeling is costly


In [ ]:
# Demo 3: VRAM back-of-envelope
def vram_gb(frames, h, w, bytes_per_activation=2, activation_factor=40):
    # wildly approximate teaching model — not a profiler
    elems = frames * h * w * 4  # latent channels proxy
    return elems * bytes_per_activation * activation_factor / (1024**3)

print(round(vram_gb(8, 64, 64), 2), "GB proxy")
print(round(vram_gb(24, 64, 64), 2), "GB proxy")


In [ ]:
# Demo 4: architecture picker
def pick(arch_needs: dict) -> str:
    if arch_needs.get("stream"):
        return "autoregressive_or_chunked"
    if arch_needs.get("max_quality") and arch_needs.get("budget") == "high":
        return "cascaded_dit_stack"
    if arch_needs.get("controllability") == "high":
        return "spatiotemporal_diffusion_with_controls"
    return "factorized_latent_diffusion"

print(pick({"controllability": "high"}))
print(pick({"stream": True}))
print(pick({"max_quality": True, "budget": "high"}))


### Try it yourself — Architectures

1. Recompute factorized ratios for 768×768 patch 8.
2. Draft a 2-stage cascade: 12fps base → 24fps interpolator responsibilities.
3. List 5 dataset defects that break temporal learning (jump cuts, burned subs, etc.).


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `inflated 3D` | Image conv/attn extended with a temporal dimension |
| `FVD` | Fréchet Video Distance — distributional video metric |
| `cascade` | Multi-stage generate-then-enhance pipeline |
| `factorized attention` | Separate spatial and temporal attention passes |


### Workshop — Parameter journal — Video Architectures

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Video Architectures
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Video Architectures

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Video Architectures
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Video Architectures

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Video Architectures
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Video Architectures

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Video Architectures
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Video Architectures

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Video Architectures
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Video Architectures

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Video Architectures
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Video Architectures

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Video Architectures
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


### Workshop — Self-check — Video Architectures

Run this checklist printer and fill it honestly before moving on.


In [ ]:
# Workshop 8 — Video Architectures
for i,x in enumerate(['restated objectives','ran demos','named pitfall','named metric'],1):
    print(f'{i}. [ ] {x}')


## Key Takeaways

- Video architectures trade full spatiotemporal modeling vs factorization
- Training signals and data quality dominate clever layers
- Scaling hits memory, eval, and safety walls early
